# Install requirements and dependencies

In [ ]:
PROJECT_PATH = "/home/jupyter-gabriel/projects/pricing_automation/pricing-automation/etl_pipeline"

In [ ]:
%pip install -r "$PROJECT_PATH/requirements.txt"

## Load kedro with magic command

This will create 3 objects available in the enviroment:
1. session (a kedro.framework.session object which is capable of creating new sessions, running pipelines or nodes)
2. context (a kedro.framework.context object which contains, among other specification, the resolve catalog)
3. catalog (an object capable of loading and saving the catalog entries defined in the catalog.yml)


In [ ]:
%load_ext kedro.ipython

Change the notebook to the project path and reload the project from there

In [ ]:
%cd $PROJECT_PATH
%reload_kedro .

## (Optional) Load custom functions
If you want to manually test the functions you build load them

You can manually run each step and take advantage of the Kedro Catalog Utility to load and save datasets specified in the catalog

OR, you can get rid of Kedro entirely by making manual loadings and savings knowing that each function requires its own inputs and outputs

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from pricing_automation.pipelines.utils import *
from pricing_automation.pipelines.a01_aoi_period import *
from pricing_automation.pipelines.n01_extract_data import *
from pricing_automation.pipelines.n02_process_data import *
from pricing_automation.pipelines.n03_create_triggers import *

# Create a Session

This lets you set quick parameters to overrride the ones in the yml files

### Step 0: Define parameters to override at runtime

In [23]:
%reload_kedro
runtime_params = {
    'country': 'colombia', # Local folder name to store data 
    'lead_id': 'hevelma_seguros', # Lead name
    'provider': 'UCSB', # Data provider, can be 'ERA5' or 'UCSB'
    'field': 'prcp', # Short variable name, can be 'swc', 'prcp', 'tmin', 'tmax'
    'start_year': 2006, # (optional) Use this to override the initial year of data
    'end_year': 2025, # (optional) Use this to override the ending year of data
}
session_trial = session.create(
    runtime_params = runtime_params
)
# (Optional) Load the catalog with the context provided earlier for quick loads
catalog_trial = session_trial.load_context().catalog 

[05/01/26 17:40:05] INFO     Resolved project path as:                                              ]8;id=565059;file:///home/jupyter-gabriel/projects/pricing_automation/.venv_pricing/lib/python3.10/site-packages/kedro/ipython/__init__.py\__init__.py]8;;\:]8;id=755175;file:///home/jupyter-gabriel/projects/pricing_automation/.venv_pricing/lib/python3.10/site-packages/kedro/ipython/__init__.py#181\181]8;;\
                             /home/jupyter-gabriel/projects/pricing_automation/pricing-automation/e                
                             tl_pipeline.                                                                          
                             To set a different path, run '%reload_kedro <project_root>'                           

                    INFO     Kedro is sending anonymous usage data with the sole purpose of improving ]8;id=948695;file:///home/jupyter-gabriel/projects/pricing_automation/.venv_pricing/lib/python3.10/site-packages/kedro_telemetry/plugin.py\plugin.py]8;;\:]8;id=916605;file:///home/jupyter-gabriel/projects/pricing_automation/.venv_pricing/lib/python3.10/site-packages/kedro_telemetry/plugin.py#242\242]8;;\
                             the product. No personal data or IP addresses are stored on our side. To              
                             opt out, set the `KEDRO_DISABLE_TELEMETRY` or `DO_NOT_TRACK` environment              
                             variables, or create a `.telemetry` file in the current working                       
                             directory with the contents `consent: false`. To hide this message,                   
                             explicitly grant or deny consent. Read more at                                        
                             https://docs.kedro.org/en/stable/about/telemetry/                                     

[05/01/26 17:40:06] INFO     Kedro project pricing_automation                                       ]8;id=244968;file:///home/jupyter-gabriel/projects/pricing_automation/.venv_pricing/lib/python3.10/site-packages/kedro/ipython/__init__.py\__init__.py]8;;\:]8;id=185992;file:///home/jupyter-gabriel/projects/pricing_automation/.venv_pricing/lib/python3.10/site-packages/kedro/ipython/__init__.py#147\147]8;;\

                    INFO     Defined global variable 'context', 'session', 'catalog' and            ]8;id=969852;file:///home/jupyter-gabriel/projects/pricing_automation/.venv_pricing/lib/python3.10/site-packages/kedro/ipython/__init__.py\__init__.py]8;;\:]8;id=505184;file:///home/jupyter-gabriel/projects/pricing_automation/.venv_pricing/lib/python3.10/site-packages/kedro/ipython/__init__.py#148\148]8;;\
                             'pipelines'                                                                           

                    INFO     Registered line magic 'run_viz'                                        ]8;id=346002;file:///home/jupyter-gabriel/projects/pricing_automation/.venv_pricing/lib/python3.10/site-packages/kedro/ipython/__init__.py\__init__.py]8;;\:]8;id=525553;file:///home/jupyter-gabriel/projects/pricing_automation/.venv_pricing/lib/python3.10/site-packages/kedro/ipython/__init__.py#154\154]8;;\

                    INFO     Kedro is sending anonymous usage data with the sole purpose of improving ]8;id=58430;file:///home/jupyter-gabriel/projects/pricing_automation/.venv_pricing/lib/python3.10/site-packages/kedro_telemetry/plugin.py\plugin.py]8;;\:]8;id=649468;file:///home/jupyter-gabriel/projects/pricing_automation/.venv_pricing/lib/python3.10/site-packages/kedro_telemetry/plugin.py#242\242]8;;\
                             the product. No personal data or IP addresses are stored on our side. To              
                             opt out, set the `KEDRO_DISABLE_TELEMETRY` or `DO_NOT_TRACK` environment              
                             variables, or create a `.telemetry` file in the current working                       
                             directory with the contents `consent: false`. To hide this message,                   
                             explicitly grant or deny consent. Read more at                                        
                             https://docs.kedro.org/en/stable/about/telemetry/                                     

Now you can call and load any entry in the Catalog, provided it exists, by running:

__catalog_trial.load('namespace.entry')__

### Step 1. Get Area Of Interest

It either creates a bounding box area with the parameters in __params.yml__ (provided the _override_ parameter is set to __true__

Otherwise it will attempt to create a bounding box from the _gdf_request_ entry in the __catalog.yml__)

You can use a large geometry and subset it on the fly. To do this use the _include_ and _exclude_ parameter in _params_s_ in the  __params.yml__ file. You should provide a dictionary of the form 

* 'column': ['value1', 'value2']

E.g.: For Argentina the Province of Buenos Aires can be filtered with 'depto' = '06' so use:

* 'include': {'depto': ['06']}

The same logic applies if it is easier to exclude some values. E.g.: Excluding the Province of Buenos Aires

* 'exclude': {'depto': ['06']}

In [ ]:
%reload_kedro 
# reload_kedro is optional, but use it if you modify any part of the project.
# This way, it reloads the project before running anything, e.g. changing a parameter.
session.create(
    runtime_params=runtime_params
).run(
    #tags=['extract']
    node_names = ['get_aoi']
)

The previous cell run a single node with its name, these can be found in: 

___src/pricing_automation/pipelines/pipeline.py___

There, you will also see tags under each entry, this groups some nodes that can or need to be run in sequence. We can specify a list of tags to run:

E.g.
session.run(tags=['extract'])

Available tags are the following:

1. 'extract': runs nodes _get_aoi_ and _extract_data_
2. 'process': runs nodes _process_data_request_ and _summarize_processed_data_
3. 'triggers': runs nodes _generate_triggers_
4. 'aep': runs nodes _run_bootstrap_aep_
5. 'pricing' runs nodes _run_bootstrap_aep_, _run_pricing_quote_ and _plot_aep_

You can run either of these nodes independently

E.g.
session.run(node_names=['namespace.get_aoi'])

___Namespaces___ are a way of running diffent pipelines or branches of pipelines under the same project.

For example, this project registered 3 namespaces: 'swc', 'prcp', 'temp' 

The only difference between them is in the _process_data_request_ node that performs different operations in each of them.

(in the __pipeline_registry.py__ file)

### Step 2. Extract data

In [ ]:
%reload_kedro
session.create(
    runtime_params = runtime_params
).run(
    tags=['extract']
)

### Step 3. Process data

In [24]:
%reload_kedro
session.create(
    runtime_params = runtime_params
).run(
    tags=['process']
    #node_names = ['swc.get_aoi']
)

[05/01/26 17:40:42] INFO     Resolved project path as:                                              ]8;id=26044;file:///home/jupyter-gabriel/projects/pricing_automation/.venv_pricing/lib/python3.10/site-packages/kedro/ipython/__init__.py\__init__.py]8;;\:]8;id=990862;file:///home/jupyter-gabriel/projects/pricing_automation/.venv_pricing/lib/python3.10/site-packages/kedro/ipython/__init__.py#181\181]8;;\
                             /home/jupyter-gabriel/projects/pricing_automation/pricing-automation/e                
                             tl_pipeline.                                                                          
                             To set a different path, run '%reload_kedro <project_root>'                           

[05/01/26 17:40:43] INFO     Kedro is sending anonymous usage data with the sole purpose of improving ]8;id=529632;file:///home/jupyter-gabriel/projects/pricing_automation/.venv_pricing/lib/python3.10/site-packages/kedro_telemetry/plugin.py\plugin.py]8;;\:]8;id=553548;file:///home/jupyter-gabriel/projects/pricing_automation/.venv_pricing/lib/python3.10/site-packages/kedro_telemetry/plugin.py#242\242]8;;\
                             the product. No personal data or IP addresses are stored on our side. To              
                             opt out, set the `KEDRO_DISABLE_TELEMETRY` or `DO_NOT_TRACK` environment              
                             variables, or create a `.telemetry` file in the current working                       
                             directory with the contents `consent: false`. To hide this message,                   
                             explicitly grant or deny consent. Read more at                                        
                             https://docs.kedro.org/en/stable/about/telemetry/                                     

                    INFO     Kedro project pricing_automation                                       ]8;id=942901;file:///home/jupyter-gabriel/projects/pricing_automation/.venv_pricing/lib/python3.10/site-packages/kedro/ipython/__init__.py\__init__.py]8;;\:]8;id=104580;file:///home/jupyter-gabriel/projects/pricing_automation/.venv_pricing/lib/python3.10/site-packages/kedro/ipython/__init__.py#147\147]8;;\

                    INFO     Defined global variable 'context', 'session', 'catalog' and            ]8;id=984807;file:///home/jupyter-gabriel/projects/pricing_automation/.venv_pricing/lib/python3.10/site-packages/kedro/ipython/__init__.py\__init__.py]8;;\:]8;id=174145;file:///home/jupyter-gabriel/projects/pricing_automation/.venv_pricing/lib/python3.10/site-packages/kedro/ipython/__init__.py#148\148]8;;\
                             'pipelines'                                                                           

                    INFO     Registered line magic 'run_viz'                                        ]8;id=682414;file:///home/jupyter-gabriel/projects/pricing_automation/.venv_pricing/lib/python3.10/site-packages/kedro/ipython/__init__.py\__init__.py]8;;\:]8;id=462398;file:///home/jupyter-gabriel/projects/pricing_automation/.venv_pricing/lib/python3.10/site-packages/kedro/ipython/__init__.py#154\154]8;;\

...

[05/01/26 17:44:10] INFO     Saving data to df_pixels (ParquetDataset)...                      ]8;id=733932;file:///home/jupyter-gabriel/projects/pricing_automation/.venv_pricing/lib/python3.10/site-packages/kedro/io/data_catalog.py\data_catalog.py]8;;\:]8;id=464972;file:///home/jupyter-gabriel/projects/pricing_automation/.venv_pricing/lib/python3.10/site-packages/kedro/io/data_catalog.py#1008\1008]8;;\

                    INFO     Completed node: summarize_processed_data                                 ]8;id=648863;file:///home/jupyter-gabriel/projects/pricing_automation/.venv_pricing/lib/python3.10/site-packages/kedro/runner/runner.py\runner.py]8;;\:]8;id=12732;file:///home/jupyter-gabriel/projects/pricing_automation/.venv_pricing/lib/python3.10/site-packages/kedro/runner/runner.py#245\245]8;;\

                    INFO     Completed 2 out of 2 tasks                                               ]8;id=716218;file:///home/jupyter-gabriel/projects/pricing_automation/.venv_pricing/lib/python3.10/site-packages/kedro/runner/runner.py\runner.py]8;;\:]8;id=62293;file:///home/jupyter-gabriel/projects/pricing_automation/.venv_pricing/lib/python3.10/site-packages/kedro/runner/runner.py#246\246]8;;\

                    INFO     Pipeline execution completed successfully in 206.7 sec.                  ]8;id=657039;file:///home/jupyter-gabriel/projects/pricing_automation/.venv_pricing/lib/python3.10/site-packages/kedro/runner/runner.py\runner.py]8;;\:]8;id=378864;file:///home/jupyter-gabriel/projects/pricing_automation/.venv_pricing/lib/python3.10/site-packages/kedro/runner/runner.py#119\119]8;;\


{
    'df_cluster': kedro_datasets.pandas.parquet_dataset.ParquetDataset(filepath=PurePosixPath('suyana-pricing/colombia/hevelma_seguros/outputs/UCSB_prcp_daily_indices.parquet'), protocol='s3', load_args={}, save_args={'compression': 'GZIP'}),
    'df_pixels': kedro_datasets.pandas.parquet_dataset.ParquetDataset(filepath=PurePosixPath('suyana-pricing/colombia/hevelma_seguros/features/UCSB_prcp_pixel_assignment.parquet'), protocol='s3', load_args={}, save_args={}),
    'ds_climatology': pricing_automation.datasets_registry.XarrayZarrDataset(filepath=PurePosixPath('suyana-pricing/colombia/hevelma_seguros/features/UCSB_prcp_climatology.zarr'), protocol='s3', load_args={'consolidated': True}, save_args={'consolidated': True, 'mode': 'w', 'append_dim': 'location_id'})
}

### Step 4. Create triggers

In [25]:
%reload_kedro
session.create(
    runtime_params = runtime_params
).run(
    tags=['triggers']
    #node_names = ['swc.get_aoi']
)

[05/01/26 17:44:16] INFO     Resolved project path as:                                              ]8;id=832331;file:///home/jupyter-gabriel/projects/pricing_automation/.venv_pricing/lib/python3.10/site-packages/kedro/ipython/__init__.py\__init__.py]8;;\:]8;id=590016;file:///home/jupyter-gabriel/projects/pricing_automation/.venv_pricing/lib/python3.10/site-packages/kedro/ipython/__init__.py#181\181]8;;\
                             /home/jupyter-gabriel/projects/pricing_automation/pricing-automation/e                
                             tl_pipeline.                                                                          
                             To set a different path, run '%reload_kedro <project_root>'                           

                    INFO     Kedro is sending anonymous usage data with the sole purpose of improving ]8;id=989357;file:///home/jupyter-gabriel/projects/pricing_automation/.venv_pricing/lib/python3.10/site-packages/kedro_telemetry/plugin.py\plugin.py]8;;\:]8;id=417089;file:///home/jupyter-gabriel/projects/pricing_automation/.venv_pricing/lib/python3.10/site-packages/kedro_telemetry/plugin.py#242\242]8;;\
                             the product. No personal data or IP addresses are stored on our side. To              
                             opt out, set the `KEDRO_DISABLE_TELEMETRY` or `DO_NOT_TRACK` environment              
                             variables, or create a `.telemetry` file in the current working                       
                             directory with the contents `consent: false`. To hide this message,                   
                             explicitly grant or deny consent. Read more at                                        
                             https://docs.kedro.org/en/stable/about/telemetry/                                     

                    INFO     Kedro project pricing_automation                                       ]8;id=342522;file:///home/jupyter-gabriel/projects/pricing_automation/.venv_pricing/lib/python3.10/site-packages/kedro/ipython/__init__.py\__init__.py]8;;\:]8;id=48700;file:///home/jupyter-gabriel/projects/pricing_automation/.venv_pricing/lib/python3.10/site-packages/kedro/ipython/__init__.py#147\147]8;;\

                    INFO     Defined global variable 'context', 'session', 'catalog' and            ]8;id=728873;file:///home/jupyter-gabriel/projects/pricing_automation/.venv_pricing/lib/python3.10/site-packages/kedro/ipython/__init__.py\__init__.py]8;;\:]8;id=355358;file:///home/jupyter-gabriel/projects/pricing_automation/.venv_pricing/lib/python3.10/site-packages/kedro/ipython/__init__.py#148\148]8;;\
                             'pipelines'                                                                           

                    INFO     Registered line magic 'run_viz'                                        ]8;id=121766;file:///home/jupyter-gabriel/projects/pricing_automation/.venv_pricing/lib/python3.10/site-packages/kedro/ipython/__init__.py\__init__.py]8;;\:]8;id=297509;file:///home/jupyter-gabriel/projects/pricing_automation/.venv_pricing/lib/python3.10/site-packages/kedro/ipython/__init__.py#154\154]8;;\

...

                    INFO     Saving data to df_policy (ParquetDataset)...                      ]8;id=590516;file:///home/jupyter-gabriel/projects/pricing_automation/.venv_pricing/lib/python3.10/site-packages/kedro/io/data_catalog.py\data_catalog.py]8;;\:]8;id=570013;file:///home/jupyter-gabriel/projects/pricing_automation/.venv_pricing/lib/python3.10/site-packages/kedro/io/data_catalog.py#1008\1008]8;;\

                    INFO     Completed node: generate_payout_policy                                   ]8;id=178594;file:///home/jupyter-gabriel/projects/pricing_automation/.venv_pricing/lib/python3.10/site-packages/kedro/runner/runner.py\runner.py]8;;\:]8;id=353919;file:///home/jupyter-gabriel/projects/pricing_automation/.venv_pricing/lib/python3.10/site-packages/kedro/runner/runner.py#245\245]8;;\

                    INFO     Completed 2 out of 2 tasks                                               ]8;id=256493;file:///home/jupyter-gabriel/projects/pricing_automation/.venv_pricing/lib/python3.10/site-packages/kedro/runner/runner.py\runner.py]8;;\:]8;id=516078;file:///home/jupyter-gabriel/projects/pricing_automation/.venv_pricing/lib/python3.10/site-packages/kedro/runner/runner.py#246\246]8;;\

                    INFO     Pipeline execution completed successfully in 3.0 sec.                    ]8;id=692438;file:///home/jupyter-gabriel/projects/pricing_automation/.venv_pricing/lib/python3.10/site-packages/kedro/runner/runner.py\runner.py]8;;\:]8;id=758024;file:///home/jupyter-gabriel/projects/pricing_automation/.venv_pricing/lib/python3.10/site-packages/kedro/runner/runner.py#119\119]8;;\


{
    'df_payouts': kedro_datasets.pandas.parquet_dataset.ParquetDataset(filepath=PurePosixPath('suyana-pricing/colombia/hevelma_seguros/outputs/UCSB_prcp_payouts.parquet'), protocol='s3', load_args={}, save_args={'compression': 'GZIP'}),
    'df_policy': kedro_datasets.pandas.parquet_dataset.ParquetDataset(filepath=PurePosixPath('suyana-pricing/colombia/hevelma_seguros/outputs/UCSB_prcp_policy.parquet'), protocol='s3', load_args={}, save_args={'compression': 'GZIP'})
}

### Step 5. Bootstrap aep

In [32]:
%reload_kedro
session.create(
    runtime_params = runtime_params
).run(
    tags=['aep']
)

[05/01/26 17:48:01] INFO     Resolved project path as:                                              ]8;id=41647;file:///home/jupyter-gabriel/projects/pricing_automation/.venv_pricing/lib/python3.10/site-packages/kedro/ipython/__init__.py\__init__.py]8;;\:]8;id=219992;file:///home/jupyter-gabriel/projects/pricing_automation/.venv_pricing/lib/python3.10/site-packages/kedro/ipython/__init__.py#181\181]8;;\
                             /home/jupyter-gabriel/projects/pricing_automation/pricing-automation/e                
                             tl_pipeline.                                                                          
                             To set a different path, run '%reload_kedro <project_root>'                           

[05/01/26 17:48:02] INFO     Kedro is sending anonymous usage data with the sole purpose of improving ]8;id=894574;file:///home/jupyter-gabriel/projects/pricing_automation/.venv_pricing/lib/python3.10/site-packages/kedro_telemetry/plugin.py\plugin.py]8;;\:]8;id=961936;file:///home/jupyter-gabriel/projects/pricing_automation/.venv_pricing/lib/python3.10/site-packages/kedro_telemetry/plugin.py#242\242]8;;\
                             the product. No personal data or IP addresses are stored on our side. To              
                             opt out, set the `KEDRO_DISABLE_TELEMETRY` or `DO_NOT_TRACK` environment              
                             variables, or create a `.telemetry` file in the current working                       
                             directory with the contents `consent: false`. To hide this message,                   
                             explicitly grant or deny consent. Read more at                                        
                             https://docs.kedro.org/en/stable/about/telemetry/                                     

                    INFO     Kedro project pricing_automation                                       ]8;id=537477;file:///home/jupyter-gabriel/projects/pricing_automation/.venv_pricing/lib/python3.10/site-packages/kedro/ipython/__init__.py\__init__.py]8;;\:]8;id=985091;file:///home/jupyter-gabriel/projects/pricing_automation/.venv_pricing/lib/python3.10/site-packages/kedro/ipython/__init__.py#147\147]8;;\

                    INFO     Defined global variable 'context', 'session', 'catalog' and            ]8;id=232340;file:///home/jupyter-gabriel/projects/pricing_automation/.venv_pricing/lib/python3.10/site-packages/kedro/ipython/__init__.py\__init__.py]8;;\:]8;id=396280;file:///home/jupyter-gabriel/projects/pricing_automation/.venv_pricing/lib/python3.10/site-packages/kedro/ipython/__init__.py#148\148]8;;\
                             'pipelines'                                                                           

                    INFO     Registered line magic 'run_viz'                                        ]8;id=723750;file:///home/jupyter-gabriel/projects/pricing_automation/.venv_pricing/lib/python3.10/site-packages/kedro/ipython/__init__.py\__init__.py]8;;\:]8;id=45963;file:///home/jupyter-gabriel/projects/pricing_automation/.venv_pricing/lib/python3.10/site-packages/kedro/ipython/__init__.py#154\154]8;;\

...

                    INFO     Saving data to df_aep (ParquetDataset)...                         ]8;id=411173;file:///home/jupyter-gabriel/projects/pricing_automation/.venv_pricing/lib/python3.10/site-packages/kedro/io/data_catalog.py\data_catalog.py]8;;\:]8;id=381876;file:///home/jupyter-gabriel/projects/pricing_automation/.venv_pricing/lib/python3.10/site-packages/kedro/io/data_catalog.py#1008\1008]8;;\

                    INFO     Completed node: run_bootstrap_aep                                        ]8;id=721999;file:///home/jupyter-gabriel/projects/pricing_automation/.venv_pricing/lib/python3.10/site-packages/kedro/runner/runner.py\runner.py]8;;\:]8;id=308345;file:///home/jupyter-gabriel/projects/pricing_automation/.venv_pricing/lib/python3.10/site-packages/kedro/runner/runner.py#245\245]8;;\

                    INFO     Completed 1 out of 1 tasks                                               ]8;id=712800;file:///home/jupyter-gabriel/projects/pricing_automation/.venv_pricing/lib/python3.10/site-packages/kedro/runner/runner.py\runner.py]8;;\:]8;id=768463;file:///home/jupyter-gabriel/projects/pricing_automation/.venv_pricing/lib/python3.10/site-packages/kedro/runner/runner.py#246\246]8;;\

                    INFO     Pipeline execution completed successfully in 0.6 sec.                    ]8;id=709449;file:///home/jupyter-gabriel/projects/pricing_automation/.venv_pricing/lib/python3.10/site-packages/kedro/runner/runner.py\runner.py]8;;\:]8;id=351558;file:///home/jupyter-gabriel/projects/pricing_automation/.venv_pricing/lib/python3.10/site-packages/kedro/runner/runner.py#119\119]8;;\


{
    'df_aep': kedro_datasets.pandas.parquet_dataset.ParquetDataset(filepath=PurePosixPath('suyana-pricing/colombia/hevelma_seguros/outputs/aep_curves.parquet'), protocol='s3', load_args={}, save_args={'compression': 'GZIP'}),
    'df_annual_agg': kedro_datasets.pandas.parquet_dataset.ParquetDataset(filepath=PurePosixPath('suyana-pricing/colombia/hevelma_seguros/outputs/aggregate_annual_losses.parquet'), protocol='s3', load_args={}, save_args={'compression': 'GZIP'})
}

### Step 7. Compute pricing quote

In [33]:
%reload_kedro
session.create(
    runtime_params = runtime_params
).run(
    tags=['pricing']
)

[05/01/26 17:48:14] INFO     Resolved project path as:                                              ]8;id=958171;file:///home/jupyter-gabriel/projects/pricing_automation/.venv_pricing/lib/python3.10/site-packages/kedro/ipython/__init__.py\__init__.py]8;;\:]8;id=265062;file:///home/jupyter-gabriel/projects/pricing_automation/.venv_pricing/lib/python3.10/site-packages/kedro/ipython/__init__.py#181\181]8;;\
                             /home/jupyter-gabriel/projects/pricing_automation/pricing-automation/e                
                             tl_pipeline.                                                                          
                             To set a different path, run '%reload_kedro <project_root>'                           

                    INFO     Kedro is sending anonymous usage data with the sole purpose of improving ]8;id=917231;file:///home/jupyter-gabriel/projects/pricing_automation/.venv_pricing/lib/python3.10/site-packages/kedro_telemetry/plugin.py\plugin.py]8;;\:]8;id=942445;file:///home/jupyter-gabriel/projects/pricing_automation/.venv_pricing/lib/python3.10/site-packages/kedro_telemetry/plugin.py#242\242]8;;\
                             the product. No personal data or IP addresses are stored on our side. To              
                             opt out, set the `KEDRO_DISABLE_TELEMETRY` or `DO_NOT_TRACK` environment              
                             variables, or create a `.telemetry` file in the current working                       
                             directory with the contents `consent: false`. To hide this message,                   
                             explicitly grant or deny consent. Read more at                                        
                             https://docs.kedro.org/en/stable/about/telemetry/                                     

                    INFO     Kedro project pricing_automation                                       ]8;id=389761;file:///home/jupyter-gabriel/projects/pricing_automation/.venv_pricing/lib/python3.10/site-packages/kedro/ipython/__init__.py\__init__.py]8;;\:]8;id=170318;file:///home/jupyter-gabriel/projects/pricing_automation/.venv_pricing/lib/python3.10/site-packages/kedro/ipython/__init__.py#147\147]8;;\

                    INFO     Defined global variable 'context', 'session', 'catalog' and            ]8;id=563952;file:///home/jupyter-gabriel/projects/pricing_automation/.venv_pricing/lib/python3.10/site-packages/kedro/ipython/__init__.py\__init__.py]8;;\:]8;id=85127;file:///home/jupyter-gabriel/projects/pricing_automation/.venv_pricing/lib/python3.10/site-packages/kedro/ipython/__init__.py#148\148]8;;\
                             'pipelines'                                                                           

                    INFO     Registered line magic 'run_viz'                                        ]8;id=917809;file:///home/jupyter-gabriel/projects/pricing_automation/.venv_pricing/lib/python3.10/site-packages/kedro/ipython/__init__.py\__init__.py]8;;\:]8;id=750805;file:///home/jupyter-gabriel/projects/pricing_automation/.venv_pricing/lib/python3.10/site-packages/kedro/ipython/__init__.py#154\154]8;;\

...

                    INFO     Saving data to df_pricing (ParquetDataset)...                     ]8;id=53468;file:///home/jupyter-gabriel/projects/pricing_automation/.venv_pricing/lib/python3.10/site-packages/kedro/io/data_catalog.py\data_catalog.py]8;;\:]8;id=161980;file:///home/jupyter-gabriel/projects/pricing_automation/.venv_pricing/lib/python3.10/site-packages/kedro/io/data_catalog.py#1008\1008]8;;\

                    INFO     Completed node: run_pricing_quote                                        ]8;id=634614;file:///home/jupyter-gabriel/projects/pricing_automation/.venv_pricing/lib/python3.10/site-packages/kedro/runner/runner.py\runner.py]8;;\:]8;id=847399;file:///home/jupyter-gabriel/projects/pricing_automation/.venv_pricing/lib/python3.10/site-packages/kedro/runner/runner.py#245\245]8;;\

                    INFO     Completed 3 out of 3 tasks                                               ]8;id=58599;file:///home/jupyter-gabriel/projects/pricing_automation/.venv_pricing/lib/python3.10/site-packages/kedro/runner/runner.py\runner.py]8;;\:]8;id=546995;file:///home/jupyter-gabriel/projects/pricing_automation/.venv_pricing/lib/python3.10/site-packages/kedro/runner/runner.py#246\246]8;;\

                    INFO     Pipeline execution completed successfully in 2.1 sec.                    ]8;id=660522;file:///home/jupyter-gabriel/projects/pricing_automation/.venv_pricing/lib/python3.10/site-packages/kedro/runner/runner.py\runner.py]8;;\:]8;id=590953;file:///home/jupyter-gabriel/projects/pricing_automation/.venv_pricing/lib/python3.10/site-packages/kedro/runner/runner.py#119\119]8;;\


{
    'df_pricing': kedro_datasets.pandas.parquet_dataset.ParquetDataset(filepath=PurePosixPath('suyana-pricing/colombia/hevelma_seguros/outputs/pricing_quote.parquet'), protocol='s3', load_args={}, save_args={'compression': 'GZIP'}),
    'plt_aep': kedro_datasets.matplotlib.matplotlib_dataset.MatplotlibDataset(filepath=PurePosixPath('suyana-pricing/colombia/hevelma_seguros/displays/aep_per_crop.png'), protocol='s3', save_args={'bbox_inches': 'tight', 'dpi': 150}),
    'plt_portfolio': kedro_datasets.matplotlib.matplotlib_dataset.MatplotlibDataset(filepath=PurePosixPath('suyana-pricing/colombia/hevelma_seguros/displays/aep_portfolio.png'), protocol='s3', save_args={'bbox_inches': 'tight', 'dpi': 150})
}

# Step 8. Create auxiliary plots

In [35]:
%reload_kedro
session.create(
    runtime_params = runtime_params
).run(
    tags=['viz']
)

[05/01/26 17:49:57] INFO     Resolved project path as:                                              ]8;id=678902;file:///home/jupyter-gabriel/projects/pricing_automation/.venv_pricing/lib/python3.10/site-packages/kedro/ipython/__init__.py\__init__.py]8;;\:]8;id=181702;file:///home/jupyter-gabriel/projects/pricing_automation/.venv_pricing/lib/python3.10/site-packages/kedro/ipython/__init__.py#181\181]8;;\
                             /home/jupyter-gabriel/projects/pricing_automation/pricing-automation/e                
                             tl_pipeline.                                                                          
                             To set a different path, run '%reload_kedro <project_root>'                           

[05/01/26 17:49:58] INFO     Kedro is sending anonymous usage data with the sole purpose of improving ]8;id=457202;file:///home/jupyter-gabriel/projects/pricing_automation/.venv_pricing/lib/python3.10/site-packages/kedro_telemetry/plugin.py\plugin.py]8;;\:]8;id=874196;file:///home/jupyter-gabriel/projects/pricing_automation/.venv_pricing/lib/python3.10/site-packages/kedro_telemetry/plugin.py#242\242]8;;\
                             the product. No personal data or IP addresses are stored on our side. To              
                             opt out, set the `KEDRO_DISABLE_TELEMETRY` or `DO_NOT_TRACK` environment              
                             variables, or create a `.telemetry` file in the current working                       
                             directory with the contents `consent: false`. To hide this message,                   
                             explicitly grant or deny consent. Read more at                                        
                             https://docs.kedro.org/en/stable/about/telemetry/                                     

                    INFO     Kedro project pricing_automation                                       ]8;id=116951;file:///home/jupyter-gabriel/projects/pricing_automation/.venv_pricing/lib/python3.10/site-packages/kedro/ipython/__init__.py\__init__.py]8;;\:]8;id=711078;file:///home/jupyter-gabriel/projects/pricing_automation/.venv_pricing/lib/python3.10/site-packages/kedro/ipython/__init__.py#147\147]8;;\

                    INFO     Defined global variable 'context', 'session', 'catalog' and            ]8;id=464438;file:///home/jupyter-gabriel/projects/pricing_automation/.venv_pricing/lib/python3.10/site-packages/kedro/ipython/__init__.py\__init__.py]8;;\:]8;id=659689;file:///home/jupyter-gabriel/projects/pricing_automation/.venv_pricing/lib/python3.10/site-packages/kedro/ipython/__init__.py#148\148]8;;\
                             'pipelines'                                                                           

                    INFO     Registered line magic 'run_viz'                                        ]8;id=729670;file:///home/jupyter-gabriel/projects/pricing_automation/.venv_pricing/lib/python3.10/site-packages/kedro/ipython/__init__.py\__init__.py]8;;\:]8;id=76509;file:///home/jupyter-gabriel/projects/pricing_automation/.venv_pricing/lib/python3.10/site-packages/kedro/ipython/__init__.py#154\154]8;;\

...

[05/01/26 17:50:27] INFO     Saving data to plt_variability_map (MatplotlibDataset)...         ]8;id=210928;file:///home/jupyter-gabriel/projects/pricing_automation/.venv_pricing/lib/python3.10/site-packages/kedro/io/data_catalog.py\data_catalog.py]8;;\:]8;id=599505;file:///home/jupyter-gabriel/projects/pricing_automation/.venv_pricing/lib/python3.10/site-packages/kedro/io/data_catalog.py#1008\1008]8;;\

[05/01/26 17:50:28] INFO     Completed node: plot_variability_map                                     ]8;id=265410;file:///home/jupyter-gabriel/projects/pricing_automation/.venv_pricing/lib/python3.10/site-packages/kedro/runner/runner.py\runner.py]8;;\:]8;id=969833;file:///home/jupyter-gabriel/projects/pricing_automation/.venv_pricing/lib/python3.10/site-packages/kedro/runner/runner.py#245\245]8;;\

                    INFO     Completed 4 out of 4 tasks                                               ]8;id=517653;file:///home/jupyter-gabriel/projects/pricing_automation/.venv_pricing/lib/python3.10/site-packages/kedro/runner/runner.py\runner.py]8;;\:]8;id=824956;file:///home/jupyter-gabriel/projects/pricing_automation/.venv_pricing/lib/python3.10/site-packages/kedro/runner/runner.py#246\246]8;;\

                    INFO     Pipeline execution completed successfully in 29.7 sec.                   ]8;id=119106;file:///home/jupyter-gabriel/projects/pricing_automation/.venv_pricing/lib/python3.10/site-packages/kedro/runner/runner.py\runner.py]8;;\:]8;id=747053;file:///home/jupyter-gabriel/projects/pricing_automation/.venv_pricing/lib/python3.10/site-packages/kedro/runner/runner.py#119\119]8;;\


{
    'plt_trend_map': kedro_datasets.matplotlib.matplotlib_dataset.MatplotlibDataset(filepath=PurePosixPath('suyana-pricing/colombia/hevelma_seguros/displays/trend_map.png'), protocol='s3', save_args={'bbox_inches': 'tight', 'dpi': 150}),
    'plt_variability_map': kedro_datasets.matplotlib.matplotlib_dataset.MatplotlibDataset(filepath=PurePosixPath('suyana-pricing/colombia/hevelma_seguros/displays/variability_map.png'), protocol='s3', save_args={'bbox_inches': 'tight', 'dpi': 150}),
    'plt_anomaly_timeseries': kedro_datasets.matplotlib.matplotlib_dataset.MatplotlibDataset(filepath=PurePosixPath('suyana-pricing/colombia/hevelma_seguros/displays/anomaly_timeseries.png'), protocol='s3', save_args={'bbox_inches': 'tight', 'dpi': 150}),
    'plt_trigger_freq_map': kedro_datasets.matplotlib.matplotlib_dataset.MatplotlibDataset(filepath=PurePosixPath('suyana-pricing/colombia/hevelma_seguros/displays/trigger_frequency_map.png'), protocol='s3', save_args={'bbox_inches': 'tight', 'dpi': 150

# Complete Run

There is no need to run each step part by part, Kedro handles dependencies at runtime.

So you can specify a list of tags to run or just run the whole pipeline from step 1 to step 7.

The _extract_data_ takes the longest to complete (about an hour for 30 years of data if AOI is ~ 4 by 4 degrees in size).

The rest of the steps are relatively faster (under 3 minutes each and some take seconds)

In [ ]:
%reload_kedro
session.create(
    runtime_params = runtime_params
).run(
    pipeline_name='swc',
    #tags=['extract', 'process', 'triggers', 'pricing']
)